### 01. Charger et explorer les données

In [1]:
import pandas as pd

# Charger le CSV brut
df = pd.read_csv(r"C:\Users\km_sa\DEV\m2_enedis_dpe_app\data\df_enedis_69.csv", sep=",")

df.head()

,annee,code_iris,nom_iris,numero_de_voie,indice_de_repetition,type_de_voie,libelle_de_voie,code_commune,nom_commune,segment_de_client,nombre_de_logements,consommation_annuelle_totale_de_l_adresse_mwh,consommation_annuelle_moyenne_par_site_de_l_adresse_mwh,consommation_annuelle_moyenne_de_la_commune_mwh,adresse,code_epci,code_departement,code_region,tri_des_adresses
0,2018,690400101,Centre,46,NaN,AVENUE,DE MONTLOUIS,69040,CHAMPAGNE-AU-MONT-D'OR,RESIDENTIEL,37,123.347,3.334,4.391,46 AVENUE DE MONTLOUIS,200046977,69,84,468802
1,2018,690400101,Centre,34,NaN,AVENUE,DE MONTLOUIS,69040,CHAMPAGNE-AU-MONT-D'OR,RESIDENTIEL,12,62.830,5.236,4.391,34 AVENUE DE MONTLOUIS,200046977,69,84,468804
2,2018,690400102,Contour,13,NaN,RUE,JEAN CLAUDE BARTET,69040,CHAMPAGNE-AU-MONT-D'OR,RESIDENTIEL,16,103.763,6.485,4.391,13 RUE JEAN CLAUDE BARTET,200046977,69,84,468810
3,2018,690400101,Centre,8,NaN,RUE,JOANNES CHOL,69040,CHAMPAGNE-AU-MONT-D'OR,RESIDENTIEL,12,27.896,2.325,4.391,8 RUE JOANNES CHOL,200046977,69,84,468814
4,2018,690400101,Centre,10,B,AVENUE,LANESSAN,69040,CHAMPAGNE-AU-MONT-D'OR,RESIDENTIEL,18,42.015,2.334,4.391,10 B AVENUE LANESSAN,200046977,69,84,468822


In [2]:
colonnes_utiles = [
    "annee",
    "nom_commune",
    "segment_de_client",
    "nombre_de_logements",
    "consommation_annuelle_totale_de_l_adresse_mwh",
    "consommation_annuelle_moyenne_par_site_de_l_adresse_mwh",
    "consommation_annuelle_moyenne_de_la_commune_mwh",
]

df = df[colonnes_utiles].dropna(subset=["nom_commune"])
df["nom_commune"] = df["nom_commune"].str.strip().str.upper()


### 02. Agréger par commune

In [3]:
agg_df = (
    df.groupby("nom_commune")
    .agg(
        {
            "consommation_annuelle_totale_de_l_adresse_mwh": "sum",
            "consommation_annuelle_moyenne_par_site_de_l_adresse_mwh": "mean",
            "consommation_annuelle_moyenne_de_la_commune_mwh": "mean",
            "nombre_de_logements": "sum"
        }
    )
    .reset_index()
)

agg_df.rename(
    columns={
        "consommation_annuelle_totale_de_l_adresse_mwh": "conso_totale_mwh",
        "consommation_annuelle_moyenne_par_site_de_l_adresse_mwh": "conso_moy_site_mwh",
        "consommation_annuelle_moyenne_de_la_commune_mwh": "conso_moy_commune_mwh",
        "nombre_de_logements": "nb_logements",
    },
    inplace=True
)

agg_df.head(10)

,nom_commune,conso_totale_mwh,conso_moy_site_mwh,conso_moy_commune_mwh,nb_logements
0,ALBIGNY-SUR-SAONE,818.498,2.949111,4.704000,319
1,ALBIGNY-SUR-SAÔNE,5611.488,2.696426,4.539971,2422
2,AMPLEPUIS,2674.603,1.831160,4.135630,1494
3,AMPUIS,1397.587,2.978172,6.102448,445
4,ANSE,10806.683,2.680710,5.110584,4240
5,ARBRESLE,2971.734,3.478039,4.916000,885
6,ARNAS,4765.961,2.392609,5.459200,2419
7,BEAUJEU,533.423,3.571308,4.840000,146
8,BEAUVALLON,837.221,3.484913,7.220609,250
9,BELLEVILLE,3408.553,2.905722,4.182000,1211


### 03. indicateur

In [4]:
agg_df["classe_conso"] = pd.qcut(
    agg_df["conso_moy_commune_mwh"],
    q=5,
    labels=["très faible", "faible", "moyenne", "élevée", "très élevée"]
)


In [5]:
agg_df.to_csv(
    r"C:\Users\km_sa\DEV\m2_enedis_dpe_app\data\conso_communes_rhone.csv",
    index=False,
    encoding="utf-8"
)
print("Fichier sauvegardé : conso_communes_rhone.csv")

Fichier sauvegardé : conso_communes_rhone.csv
